In [1]:
%pip install -qU --no-deps langchain-groq langchain-google-genai
%pip install -qU langchain langchain-core "groq<1.0.0"

In [2]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    groq_api_key = userdata.get("new_api")   # <- your secret name, this can stay "api_new"
except Exception:
    groq_api_key = None

if not groq_api_key:
    groq_api_key = getpass("Enter your GROQ_API_KEY: ")

os.environ["GROQ_API_KEY"] = groq_api_key   # <- this MUST be exactly "GROQ_API_KEY", don't rename this one

try:
    google_api_key = userdata.get("Gemini_API")   # <- your secret name, keep whatever you named it
except Exception:
    google_api_key = None

if not google_api_key:
    google_api_key = getpass("Enter your GOOGLE_API_KEY: ")

os.environ["GOOGLE_API_KEY"] = google_api_key   # <- this MUST be exactly "GOOGLE_API_KEY", don't rename this one

print("API keys configured successfully.")

API keys configured successfully.


In [5]:
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0,
)

result = gemini_llm.invoke("Say hello in one short sentence.")
print("Gemini test response:", result.content)

Gemini test response: [{'type': 'text', 'text': 'Hello, it is nice to meet you!', 'extras': {'signature': 'EjQKMgERTTIPlIv80w8kkb4PGt6d7Na/zIJ0TtUylR6pFroLk0hl/cU2WNepr9D4gACeh8B+'}}]


In [6]:
result = gemini_llm.invoke("Say hello in one short sentence.")

if isinstance(result.content, list):
    gemini_text = result.content[0]["text"]
else:
    gemini_text = result.content

print("Gemini test response:", gemini_text)

Gemini test response: Hello, it is nice to meet you!


In [7]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

groq_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0,
)

def get_text(result):
    """Helper: Gemini returns a list of blocks, Groq returns a plain string."""
    if isinstance(result.content, list):
        return result.content[0]["text"]
    return result.content

print("Groq and Gemini models are ready.")

Groq and Gemini models are ready.


In [8]:
question = "What is the capital of Italy?"

groq_result = groq_llm.invoke(question)
gemini_result = gemini_llm.invoke(question)

print("Groq:", get_text(groq_result))
print("Gemini:", get_text(gemini_result))

Groq: The capital of Italy is Rome.
Gemini: The capital of Italy is Rome.


In [9]:
prompt = "Write a 4-line poem about the ocean."

for name, model in [("Groq", ChatGroq(model="llama-3.1-8b-instant", temperature=1.2)),
                    ("Gemini", ChatGoogleGenerativeAI(model="gemini-flash-lite-latest", temperature=1.2))]:
    result = model.invoke(prompt)
    print(f"--- {name} ---")
    print(get_text(result))
    print()

--- Groq ---
Waves caress the sandy shore,
A soothing melody evermore.
Deep blue waters, endless sea,
A mystery that's yet to be.

--- Gemini ---
Endless waves embrace the sand,
Where blue meets sky and distant land.
A rhythmic pulse of salt and tide,
With hidden worlds tucked deep inside.



In [10]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(
        content="You are a helpful PGD teacher. Explain concepts in easy words and give one practical example."
    ),
    HumanMessage(
        content="What is the recipe to make Biryani?"
    ),
]

groq_result = groq_llm.invoke(messages)
gemini_result = gemini_llm.invoke(messages)

print("--- Groq ---")
print(get_text(groq_result))
print()
print("--- Gemini ---")
print(get_text(gemini_result))

--- Groq ---
Biryani is a popular South Asian dish made with a mixture of spices, basmati rice, and marinated meat or vegetables. Here's a simple recipe to make Biryani:

**Ingredients:**

For the rice:
- 2 cups basmati rice
- 4 cups water
- 1 tablespoon ghee or oil
- Salt to taste

For the marinade:
- 1 pound boneless chicken or beef (or vegetables like carrots, peas, and cauliflower)
- 1/2 cup yogurt
- 2 tablespoons lemon juice
- 1 teaspoon garam masala powder
- 1 teaspoon cumin powder
- 1 teaspoon coriander powder
- 1/2 teaspoon cayenne pepper (optional)
- Salt to taste

For the spice blend:
- 2 tablespoons coriander seeds
- 1 tablespoon cumin seeds
- 1 tablespoon cinnamon powder
- 1 tablespoon cardamom powder
- 1 tablespoon cloves powder
- 1 tablespoon saffron threads (optional)

**Instructions:**

1. **Prepare the marinade:** In a bowl, mix together the yogurt, lemon juice, garam masala powder, cumin powder, coriander powder, cayenne pepper (if using), and salt. Add the chicken or

In [11]:
questions = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is Generative AI?",
]

for question in questions:
    groq_answer = get_text(groq_llm.invoke(question))
    gemini_answer = get_text(gemini_llm.invoke(question))

    print(f"Question: {question}")
    print(f"Groq: {groq_answer}")
    print(f"Gemini: {gemini_answer}")
    print("-" * 70)

Question: What is Artificial Intelligence?
Groq: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

AI technology is based on the principle of creating algorithms that can process data, identify patterns, and make decisions without being explicitly programmed for each specific task. This allows AI systems to adapt and improve over time, much like humans do.

There are several key characteristics of AI:

1. **Machine Learning**: AI systems can learn from data and improve their performance over time.
2. **Reasoning**: AI systems can draw conclusions and make decisions based on the data they have been trained on.
3. **Problem-Solving**: AI systems can identify and solve complex problems.
4. **Natural Language Processing**: AI systems can understand and generate human lan

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert teacher for {domain}. "
            "Explain the answer in simple language using bullet points."
        ),
        (
            "human",
            "Explain {topic} and give one practical example."
        ),
    ]
)

formatted_messages = prompt_template.invoke(
    {
        "domain": "Generative AI",
        "topic": "Role in Health Sector",
    }
)

print("--- Groq ---")
print(get_text(groq_llm.invoke(formatted_messages)))
print()
print("--- Gemini ---")
print(get_text(gemini_llm.invoke(formatted_messages)))

--- Groq ---
**Role of Generative AI in the Health Sector:**

* **Medical Imaging Analysis**: Generative AI can analyze medical images such as X-rays, CT scans, and MRIs to help doctors diagnose diseases more accurately and quickly.
* **Personalized Medicine**: Generative AI can help create personalized treatment plans for patients based on their genetic profiles, medical history, and lifestyle.
* **Predictive Analytics**: Generative AI can analyze large amounts of medical data to predict patient outcomes, identify high-risk patients, and prevent hospital readmissions.
* **Clinical Decision Support**: Generative AI can provide doctors with real-time clinical decision support, suggesting the best course of treatment based on the latest medical research and guidelines.
* **Virtual Nursing Assistants**: Generative AI can power virtual nursing assistants that can help patients with routine tasks, such as medication reminders and appointment scheduling.

**Practical Example:**

**Example:**

In [14]:
def ask_model(question, provider="groq", temperature=0):
    if provider == "groq":
        model = ChatGroq(model="llama-3.1-8b-instant", temperature=temperature)
    else:
        model = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest", temperature=temperature)

    return get_text(model.invoke(question))


print(ask_model("Explain the difference between discriminative AI and generative AI.", provider="groq"))
print()
print(ask_model("Explain the difference between discriminative AI and generative AI.", provider="gemini"))

Discriminative AI and generative AI are two fundamental approaches in the field of artificial intelligence (AI). The primary difference between them lies in their objectives and the types of tasks they are designed to perform.

**Discriminative AI:**

Discriminative AI models are designed to make predictions or classify data based on existing patterns and relationships. Their primary goal is to identify and distinguish between different classes or categories. These models typically use supervised learning techniques, where the model is trained on labeled data to learn the mapping between inputs and outputs.

Examples of discriminative AI tasks include:

1. Image classification: Identifying objects or scenes in images.
2. Speech recognition: Transcribing spoken words into text.
3. Sentiment analysis: Determining the emotional tone of text or speech.
4. Spam detection: Identifying spam emails or messages.

Discriminative AI models are typically trained to optimize a specific objective fu

In [15]:
%pip install -qU --no-deps sentence-transformers
%pip install -qU scikit-learn

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [17]:
text = "Islamabad is the capital of Pakistan."

vector = embedding_model.embed_query(text)

print("Text:", text)
print("Vector length:", len(vector))
print("First 10 values:", vector[:10])

Text: Islamabad is the capital of Pakistan.
Vector length: 384
First 10 values: [0.018565110862255096, 0.08578696846961975, -0.05348091945052147, 0.09558004885911942, -0.002001244807615876, -0.06132710725069046, 0.08399884402751923, -0.020370151847600937, -0.007159597240388393, 0.03674142062664032]


In [18]:
documents = [
    "Islamabad is the capital of Pakistan.",
    "Karachi is the largest city of Pakistan.",
    "Paris is the capital of France.",
]

document_vectors = embedding_model.embed_documents(documents)

print("Number of document vectors:", len(document_vectors))
print("Dimensions of each vector:", len(document_vectors[0]))
print("First 3 values of the first vector:", document_vectors[0][:3])

Number of document vectors: 3
Dimensions of each vector: 384
First 3 values of the first vector: [0.018565159291028976, 0.08578697592020035, -0.05348089337348938]


In [19]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Virat Kohli is an Indian cricketer known for aggressive batting.",
    "MS Dhoni is a former Indian captain known for calm leadership.",
    "Sachin Tendulkar holds many international batting records.",
    "Rohit Sharma is known for elegant batting and double centuries.",
    "Jasprit Bumrah is an Indian fast bowler known for yorkers.",
]

query = "Tell me about Bumrah."

document_vectors = embedding_model.embed_documents(documents)
query_vector = embedding_model.embed_query(query)

scores = cosine_similarity([query_vector], document_vectors)[0]

best_index = int(np.argmax(scores))

print("Query:", query)
print()
print("Similarity scores:")
for document, score in zip(documents, scores):
    print(f"{score:.3f}  |  {document}")

print()
print("Best matching document:")
print(documents[best_index])
print("Similarity score:", round(float(scores[best_index]), 4))

Query: Tell me about Bumrah.

Similarity scores:
0.082  |  Virat Kohli is an Indian cricketer known for aggressive batting.
0.175  |  MS Dhoni is a former Indian captain known for calm leadership.
0.115  |  Sachin Tendulkar holds many international batting records.
0.155  |  Rohit Sharma is known for elegant batting and double centuries.
0.586  |  Jasprit Bumrah is an Indian fast bowler known for yorkers.

Best matching document:
Jasprit Bumrah is an Indian fast bowler known for yorkers.
Similarity score: 0.5864


In [20]:
def semantic_search(query, documents, top_k=3):
    document_vectors = embedding_model.embed_documents(documents)
    query_vector = embedding_model.embed_query(query)

    scores = cosine_similarity([query_vector], document_vectors)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for index in ranked_indices:
        results.append({"document": documents[index], "score": float(scores[index])})

    return results


search_results = semantic_search(
    query="Who is a fast bowler?",
    documents=documents,
    top_k=3,
)

for rank, item in enumerate(search_results, start=1):
    print(f"{rank}. Score: {item['score']:.4f}")
    print(item["document"])
    print()

1. Score: 0.5394
Jasprit Bumrah is an Indian fast bowler known for yorkers.

2. Score: 0.4869
Rohit Sharma is known for elegant batting and double centuries.

3. Score: 0.4712
Sachin Tendulkar holds many international batting records.



In [21]:
from langchain_core.messages import HumanMessage, SystemMessage

knowledge_base = [
    "Islamabad became the capital of Pakistan in the 1960s.",
    "Karachi is Pakistan's largest city and a major commercial centre.",
    "Lahore is known for its history, culture, food, and educational institutions.",
    "Peshawar is the capital city of Khyber Pakhtunkhwa.",
]

user_question = "Which city is the capital of Pakistan?"

retrieved_items = semantic_search(query=user_question, documents=knowledge_base, top_k=1)
retrieved_context = retrieved_items[0]["document"]

rag_messages = [
    SystemMessage(
        content=(
            "Answer the question using only the supplied context. "
            "If the answer is not in the context, say that the context "
            "does not contain enough information."
        )
    ),
    HumanMessage(
        content=f"Context:\n{retrieved_context}\n\nQuestion:\n{user_question}"
    ),
]

groq_answer = get_text(groq_llm.invoke(rag_messages))
gemini_answer = get_text(gemini_llm.invoke(rag_messages))

print("Retrieved context:")
print(retrieved_context)
print()
print("Groq answer:", groq_answer)
print("Gemini answer:", gemini_answer)

Retrieved context:
Islamabad became the capital of Pakistan in the 1960s.

Groq answer: Islamabad is the capital of Pakistan.
Gemini answer: Islamabad is the capital of Pakistan.
